## Step 2 - Pocket dataframes, Global ID assignment & apo/holo comparison
This is the second explanatory notebook of the pipeline.

Step 1 left every replicate's `pockets/` folder full of mdpocket/ATClus output (dummy-atom PDBs, per-frame descriptor files, residue files). This notebook turns that into per-pocket dataframes, assigns Global IDs across replicates/genes/states, and compares apo vs holo pocketomes.

Two scripts, run in sequence:

1. **`pipeline/pocket_dataframes.py`** (Step 2.1) -- parses every pocket, computes the interpolated volume, orthosteric/transient/largest-pocket flags and volume category, and writes two dataframes: `all_pockets` (every pocket, every frame, every alpha-sphere coordinate) and `pocket_summary` (one row per pocket, its first frame). No Global ID here -- a pocket's Global ID depends on which subset of experiments it gets clustered against.
2. **`pipeline/global_id_and_comparison.py`** (Step 2.2) -- clusters a *chosen subset* of `all_pockets` (by state / gene / PDB ID) into Global IDs via voxel-IoU overlap, then (optionally) compares apo vs holo pocketomes for that subset.

Both scripts write every big table as *both* `.csv` and `.parquet` (`pipeline/pocket_io.py`) -- read whichever you prefer, `pocket_io.load_table()` picks parquet by default and falls back to csv.

In [ ]:
import os, sys
import importlib
import time
import warnings

sys.path.insert(0, os.path.join(os.getcwd(), 'pipeline'))  # pocket_dataframes.py & co import each other by bare name
import pocket_io
import pocket_dataframes
import global_id_and_comparison as gid
import config as conf

importlib.reload(pocket_io)
importlib.reload(pocket_dataframes)
importlib.reload(gid)
importlib.reload(conf)

warnings.filterwarnings("ignore")


Define the global variables. `SAVING_LOC` is where Step 2.1's `all_pockets` / `pocket_summary` land (default `output/meta_analysis/across_genes/`); Global ID runs each get their own subdirectory under `output/meta_analysis/` so different subsets never mix numbering (see Step 2.2 below).

In [ ]:
# Globals
VERBOSE = True
MAKE_PLOTS = True
FORMATS = ('csv', 'parquet')          # every big table is written as both
SAVING_LOC = conf.META_ANALYSIS_DIR   # output/meta_analysis/across_genes

BW_FILE_LOC = os.path.join(conf.REFERENCE_DATA_DIR, 'TAARs_numbered')
PDB_FILE_LOC = os.path.join(conf.HOLO_RESULTS_DIR, 'holo8ITF', '1')  # backbone outline for the largest/orthosteric-pocket plots


## Step 2.1 - Building the pocket dataframes
`pocket_dataframes.pocket_dirs_for()` walks `output/{apo,holo}_structures/<state><PDBID>/<rep>/pockets/` (Step 1's output) and returns every replicate that has one. `build_pocket_dataframes()` then, per pocket:

- parses the dummy-atom/descriptor/residue files (`PocketFileParser`)
- interpolates `pock_volume` across short closures -> `interpolated_pock_volume`
- classifies orthosteric/binding-site pockets (`orthosteric_filter_ligand_based.classify_binding_site`, the single source of truth for `is_orthosteric`)
- flags the largest pocket per (state, PDB ID, replicate) experiment, and `is_largest_and_orthosteric`
- classifies transient/stable pockets (`consecutive_zeros_transiency.classify_transient`)
- derives `volume_category` from the per-pocket trajectory-median volume (open frames only)

and writes `all_pockets` and `pocket_summary` (+ `orthosteric_perframe_volumes.csv`, a cheap filter of `all_pockets` that `taar_paper_figures/fig2_binding_site.py` reads directly).

In [ ]:
pocket_dirs = pocket_dataframes.pocket_dirs_for()
print(f'Found {len(pocket_dirs)} pocket directories')
pocket_dirs


In [ ]:
result = pocket_dataframes.build_pocket_dataframes(
    pocket_dirs,
    saving_loc=SAVING_LOC,
    formats=FORMATS,
    make_plots=MAKE_PLOTS,
    bw_file_loc=BW_FILE_LOC,
    pdb_file_loc=PDB_FILE_LOC,
    verbose=VERBOSE,
)
all_pockets, pocket_summary = result['all_pockets'], result['pocket_summary']
pocket_summary.head()


## Step 2.2 - Global ID assignment
A pocket's Global ID is only meaningful **within the run that produced it** -- two pockets can only share a Global ID if they were voxel-clustered together, so Global IDs from two different subsets are never comparable. `global_id_and_comparison.select_subset()` picks, in-memory from `all_pockets`, exactly the pockets that should be clustered together; `assign_global_ids()` then runs the voxel-IoU clustering (`voxel_intersection_over_union_global_id`) and writes `pocket_comparison_table` (one row per Local Pocket ID, with its Global ID and gene/replicate uniqueness annotations).

The subset is picked with plain keyword arguments, not a CLI flag -- `states=`/`genes=`/`pdb_ids=` on `select_subset()` below (`None` = no filter on that axis):

- **`states=None`** (recommended default, shown below) -- apo and holo clustered *together*, sharing one Global ID numbering. Needed if you want `run_apo_holo_comparison()` afterwards.
- **`states=['apo']`** / **`states=['holo']`** -- cluster one state alone.
- apo and holo clustered *separately* (independent numbering per run), then reconciled geometrically by `match_states()` -- see the optional cell further down; use this only if you deliberately want two separate runs reconciled after the fact rather than one shared run.

`genes=[...]` / `pdb_ids=[...]` restrict further (e.g. one gene's pockets across both states). `global_id_and_comparison.main(states=..., genes=..., pdb_ids=...)` bundles `select_subset` + `assign_global_ids` + `run_apo_holo_comparison` into one call with these same keyword arguments, if you don't need the intermediate dataframes -- the cells below just show what it does.

In [ ]:
# Re-load from disk here rather than reusing `all_pockets` in memory, so this cell also works
# standalone (e.g. re-running just Step 2.2 later against an existing Step 2.1 output).
all_pockets = pocket_io.load_table(SAVING_LOC, 'all_pockets')

GID_SAVING_LOC = os.path.join(conf.META_ANALYSIS_ROOT, 'global_ID_combined')
subset = gid.select_subset(all_pockets, states=None, genes=None, pdb_ids=None)  # None = no filter on that axis; states=None here means BOTH apo and holo, clustered together

pocket_comparison_table = gid.assign_global_ids(
    subset,
    GID_SAVING_LOC,
    formats=FORMATS,
    make_plots=MAKE_PLOTS,
)
pocket_comparison_table.head()


### Apo vs holo pocketome comparison
Reuses `taar_paper_figures/pocketome_metrics.py` (allosteric pocket count/volume/size-class deltas + Jensen-Shannon distance) and `taar_paper_figures/fig2_binding_site.py` (orthosteric site volume shift) against the subset just clustered above -- requires both states in that subset, i.e. `--subset combined` or `both`.

In [ ]:
# run_apo_holo_comparison reads pocket_summary (not pocket_comparison_table) for the metrics themselves --
# they're per (PDB ID, state, replicate), not Global-ID-dependent -- but takes the same states/genes/pdb_ids
# filter so it's comparing the same subset you just clustered.
pocket_summary_subset = gid.select_subset(pocket_summary, states=None, genes=None, pdb_ids=None)
apo_holo_summary = gid.run_apo_holo_comparison(pocket_summary_subset, SAVING_LOC, saving_loc=GID_SAVING_LOC)
apo_holo_summary.head()


### Optional: apo-only and holo-only, reconciled afterwards
The alternative to `combined`: cluster each state on its own (independent Global ID numbering per run), then recover the apo<->holo correspondence geometrically from where each global pocket actually sits. Skip this cell unless you deliberately want two separate runs.

In [ ]:
# apo_loc = os.path.join(conf.META_ANALYSIS_ROOT, 'global_ID_apo')
# holo_loc = os.path.join(conf.META_ANALYSIS_ROOT, 'global_ID_holo')
# apo_table = gid.assign_global_ids(gid.select_subset(all_pockets, states=['apo']), apo_loc, formats=FORMATS)
# holo_table = gid.assign_global_ids(gid.select_subset(all_pockets, states=['holo']), holo_loc, formats=FORMATS)
# mapping_loc = os.path.join(conf.META_ANALYSIS_ROOT, 'apo_holo_global_id_mapping')
# mapping_df = gid.match_states(
#     apo_table, holo_table, mapping_loc,
#     os.path.join(apo_loc, 'global_pockets_IoU_voxel.csv'),
#     os.path.join(holo_loc, 'global_pockets_IoU_voxel.csv'),
# )


## End of Step 2
By now `output/meta_analysis/` should hold, e.g.:

```
output/meta_analysis/
├── across_genes/                                  # Step 2.1 output
│   ├── all_pockets.csv, all_pockets.parquet        # every pocket, every frame, every alpha-sphere coordinate
│   ├── pocket_summary.csv, pocket_summary.parquet  # one row per pocket (first frame)
│   ├── orthosteric_perframe_volumes.csv
│   └── plots_largest/, plots_orthosteric/, plots_comparison/
└── global_ID_combined/                             # Step 2.2 output (one subdirectory per subset)
    ├── pocket_comparison_table.csv / .parquet       # one row per Local Pocket ID, with Global ID
    ├── global_pockets_IoU_voxel.csv                 # point-cloud + voxel_group_id, for QC/downstream reuse
    ├── apo_holo_pocketome_summary.csv
    ├── unique_to_gene_comparison.csv, unique_to_structure_comparison.csv, shared_in_all_gene_comparison.csv
    └── pocket_clusters_qc.html, <PDBID>_within_pdb_global_id_plot.html, upsetplot_gene_comparison.png, ...
```

For publication figures built from this output, see **`taar_paper_figures/`** (`paper_plots.py` is its single entry point).